<a href="https://colab.research.google.com/github/satyam-1605/RAG-from-basic-to-advance/blob/main/Chat_With_Multiple_Doc(pdfs%2C_docs%2C_txt%2C_pptx)_using_AstraDB_and_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain


In [ ]:
!pip install -q unstructured Cython tiktoken

In [ ]:
!pip install -q langchain-astradb

In [ ]:
!pip install -q langchain-community langchain-google-genai datasets pypdf

In [ ]:
!pip install -q pdf2image

In [ ]:
!pip install -q  pdfminer.six

In [ ]:
!pip install -q unstructured[pdf]

In [ ]:
import os
from getpass import getpass

from datasets import (
    load_dataset,
)
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredPDFLoader


In [ ]:
from langchain_classic.indexes import VectorstoreIndexCreator

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings

In [ ]:
import os
from google.colab import userdata
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY



In [ ]:
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# Using Unstructured for loading Multiple Pdfs

In [ ]:
root_dir="/content/content"

In [ ]:
pdf_folder_path = f'{root_dir}/docs/'

In [ ]:
os.listdir(pdf_folder_path)

In [ ]:
# location of the pdf file/files.
loaders = [UnstructuredPDFLoader(os.path.join(pdf_folder_path, fn)) for fn in os.listdir(pdf_folder_path)]

In [ ]:
loaders

In [ ]:
filtered_loaders = []
for fn in os.listdir(pdf_folder_path):
    if fn.lower().endswith('.pdf'):
        filtered_loaders.append(UnstructuredPDFLoader(os.path.join(pdf_folder_path, fn)))

index = VectorstoreIndexCreator(embedding=embedding).from_loaders(filtered_loaders)

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)
index.query('What is RAG?', llm=llm)

In [ ]:
index.query_with_sources('What is the main problem with large pre-trained language models that RAG tries to solve?',llm=llm)

# Pypdf loader with Multiple Pdfs.

In [ ]:
from langchain_astradb import AstraDBVectorStore

In [ ]:
from langchain_astradb import AstraDBVectorStore
ASTRA_DB_API_ENDPOINT="api_endpoint"
ASTRA_DB_APPLICATION_TOKEN="api_token"
ASTRA_DB_KEYSPACE="default_keyspace"

In [ ]:
root_dir="/content/content"
pdf_folder_path = f'{root_dir}/docs/'
pdfs=os.listdir(pdf_folder_path)

In [ ]:
pdfs

In [ ]:
data=PyPDFLoader("/content/content/docs/MachineTranslationwithAttention.pdf")

In [ ]:
data

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)

In [ ]:
data.load_and_split(text_splitter=splitter)

In [ ]:
docs=[]
for pdf_file in pdfs:
  if pdf_file.lower().endswith('.pdf'):
    data=PyPDFLoader(f"{pdf_folder_path}{pdf_file}")
    docs.append(data)

In [ ]:
all_docs = []
for loader in docs:
    all_docs.extend(loader.load_and_split(text_splitter=splitter))
docs_from_pdf = all_docs

In [ ]:
!pip install -q langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# No API key, No rate limit!
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


In [ ]:
print(f"Documents from PDF: {len(docs_from_pdf)}.")
inserted_ids_from_pdf = vstore.add_documents(docs_from_pdf)
print(f"Inserted {len(inserted_ids_from_pdf)} documents.")

In [ ]:
vstore = AstraDBVectorStore(
    embedding=embedding,
    collection_name="my_collection",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=ASTRA_DB_KEYSPACE,
)

In [ ]:
retriever = vstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
prompt_template = """
You are a helpful and knowledgeable AI assistant.
Use the provided context as the basis for your answers.
Do not make up new information - only use what is given in the context.

Your answers must be:
- Concise and to the point
- Based only on the provided context
- Well structured and easy to understand
- If the answer is not found in context,
  say "I don't have enough information to answer this question"

CONTEXT:
{context}

QUESTION: {question}

YOUR ANSWER:"""

In [ ]:
prompt_template = ChatPromptTemplate.from_template(prompt_template)

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [ ]:
chain.invoke("What BLEU score did the attention model achieve?")

# Directory loders(Chat With Multiple Doc)

In [ ]:
!sudo apt-get update

In [ ]:
!sudo apt-get install poppler-utils

In [ ]:
!sudo apt-get install libleptonica-dev tesseract-ocr libtesseract-dev python3-pil tesseract-ocr-eng tesseract-ocr-script-latn


In [ ]:
!pip install unstructured-pytesseract
!pip install tesseract-ocr

In [ ]:
!pip install "unstructured[pptx]"

In [ ]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    UnstructuredPowerPointLoader,
    PyPDFLoader,
    TextLoader
)

# PPT Loader
ppt_loader = DirectoryLoader(
    "/content/content/docs",
    glob="**/*.pptx",
    loader_cls=UnstructuredPowerPointLoader
)

# PDF Loader
pdf_loader = DirectoryLoader(
    "/content/content/docs",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

# TXT Loader
txt_loader = DirectoryLoader(
    "/content/content/docs",
    glob="**/*.txt",
    loader_cls=TextLoader
)

ppt_docs = ppt_loader.load()
pdf_docs = pdf_loader.load()
txt_docs = txt_loader.load()

all_docs = ppt_docs + pdf_docs + txt_docs

print("PPT Docs:", len(ppt_docs))
print("PDF Docs:", len(pdf_docs))
print("TXT Docs:", len(txt_docs))

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)

In [ ]:
# docs = loader.load_and_split(text_splitter=splitter)

In [ ]:
docs = splitter.split_documents(all_docs)

In [ ]:
for i, doc in enumerate(docs):
    print(f"\n--- CHUNK {i} ---")
    print(doc.page_content)

In [ ]:
for i, doc in enumerate(docs):
    print(f"\n--- CHUNK {i} ---")
    print(doc.page_content)
    print(doc.metadata)

In [ ]:
len(docs)

In [ ]:
inserted_ids = vstore.add_documents(docs)

In [ ]:
print(f"\nInserted {len(inserted_ids)} documents.")

In [ ]:
chain.invoke("common type of personal cybercrime?")

In [ ]:
chain.invoke("what is identity theft?")

In [ ]:
chain.invoke("What did the president say about Zelenskyy")